In [ ]:
library(tidyverse)

## Load data
Data needed: 011626_demographics.csv, 011626_control_condition_df.csv, 011626_adhd_condition_df.csv

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_demographics.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
demographics  <- read_csv(name_of_file_in_bucket)

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_control_condition_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
control_condition_df  <- read_csv(name_of_file_in_bucket)

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_adhd_condition_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
adhd_condition_df  <- read_csv(name_of_file_in_bucket)

## Process data for descriptive analysis

In [ ]:
# Merge ADHD and control condition_dfs
condition_df <- bind_rows(adhd_condition_df, control_condition_df)

# Filter df to retain rows with person_ids in demographics
condition_df <- condition_df %>%
    filter(person_id %in% demographics$person_id)

In [ ]:
head(condition_df)

In [ ]:
# Define functions for extracting dxs

# Define function: extract_ADHD_subtype()
extract_ADHD_subtype <- function(data, id_column, dx_column) {

  # Define list of ADHD diagnoses
    dx_list_ADHD <- c("Attention deficit hyperactivity disorder, combined type",
                      "Attention deficit hyperactivity disorder, predominantly inattentive type",
                      "Attention deficit hyperactivity disorder, predominantly hyperactive impulsive type",
                      "Attention deficit hyperactivity disorder",
                      "Adult attention deficit hyperactivity disorder",
                      "Child attention deficit disorder",
                      "Undifferentiated attention deficit disorder")
  
  # Define buckets for ADHD subtypes  
      ADHD_H <- "Attention deficit hyperactivity disorder, predominantly hyperactive impulsive type"
      ADHD_U <- c("Attention deficit hyperactivity disorder",
                  "Adult attention deficit hyperactivity disorder",
                  "Child attention deficit disorder",
                  "Undifferentiated attention deficit disorder"
      )
      ADHD_PI <- "Attention deficit hyperactivity disorder, predominantly inattentive type"
      ADHD_C <- "Attention deficit hyperactivity disorder, combined type"
  
  data %>%
    distinct() %>%
    select({{id_column}},{{dx_column}}) %>%
    group_by({{id_column}}) %>%

    mutate( ADHD = 
      case_when(
        # ADHD = "None" if no ADHD diagnosis is present
        !any({{dx_column}} %in% dx_list_ADHD) ~ "None", 
        # ADHD = "Combined" if combined diagnosis is present OR
        any({{dx_column}} %in% ADHD_C) |
          # If hyperactive and inattentive diagnoses co-occur
          (any({{dx_column}} %in% ADHD_PI) & any({{dx_column}} %in% ADHD_H)) ~ "Combined",
        # ADHD = "Inattentive" if inattentive is present
        any({{dx_column}} %in% ADHD_PI) ~ "Inattentive",
        # ADHD = "Hyperactive" if hyperactive is present
        any({{dx_column}} %in% ADHD_H) ~ "Hyperactive",
        # ADHD = "Unspecified" if only unspecified diagnoses are present
        any({{dx_column}} %in% ADHD_U) ~ "Unspecified",
        TRUE ~ "None"
      )) %>%
   summarize("ADHD_subtype" = first(ADHD), .groups = "drop")
}

# Define function: extract_substance_dependence()
extract_substance_dependence <- function(data, id_column, dx_column) {

  data %>%
    select({{id_column}}, {{dx_column}}) %>%
    distinct() %>%
    group_by({{id_column}}) %>%
    mutate(
      alcohol_dependence = str_detect({{dx_column}},
                              regex("alcohol dependence|alcoholism|alcohol abuse",
                              ignore_case = TRUE)
                            ),
      cannabis_dependence = str_detect({{dx_column}},
                              regex("cannabis dependence",
                              ignore_case = TRUE)
                            ),
      cocaine_dependence = str_detect({{dx_column}},
                              regex("cocaine dependence",
                              ignore_case = TRUE)
                            ),
      nicotine_dependence = str_detect({{dx_column}},
                              regex("nicotine dependence|tobacco dependence",
                              ignore_case = TRUE)
                            ),
      opioid_dependence = str_detect({{dx_column}},
                              regex("heroin dependence|opioid dependence",
                              ignore_case = TRUE)
                            )
    ) %>%
    select({{id_column}}, alcohol_dependence:opioid_dependence) %>%
    summarize(
      across(alcohol_dependence:opioid_dependence,
         ~any(.x)), 
         .groups = "drop")
}

In [ ]:
# Extract ADHD dxs
adhd <- extract_ADHD_subtype(condition_df, person_id, standard_concept_name)

# Extract dependence dxs
dependence <- extract_substance_dependence(condition_df, person_id, standard_concept_name)

In [ ]:
head(adhd)
head(dependence)
nrow(adhd)
nrow(dependence)

In [ ]:
# Merge with demographics_df

merge1 <- left_join(adhd, dependence, by = 'person_id')
cohort_full <- left_join(demographics, merge1, by = 'person_id')

In [ ]:
# Remove unneeded columns
cohort_full <- cohort_full %>%
    select(-distance)

In [ ]:
# Save output to bucket
my_dataframe <- cohort_full

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- '011626_cohort_full.csv'

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


## Process dfs for substance use PheWAS

In [ ]:
library(tidyverse)
# Load data
name_of_file_in_bucket <- '011626_cohort_full.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
cohort_full  <- read_csv(name_of_file_in_bucket)
head(cohort_full)

In [ ]:
# Alcohol dependence cohort for PheWAS
cohort_alcohol <- cohort_full %>%
    filter(alcohol_dependence == TRUE)
nrow(cohort_alcohol)

In [ ]:
# Cannabis dependence cohort for PheWAS
cohort_cannabis <- cohort_full %>%
    filter(cannabis_dependence == TRUE)
nrow(cohort_cannabis)

In [ ]:
# Cocaine dependence cohort for PheWAS
cohort_cocaine <- cohort_full %>%
    filter(cocaine_dependence == TRUE)
nrow(cohort_cocaine)

In [ ]:
# Nicotine dependence cohort for PheWAS
cohort_nicotine <- cohort_full %>%
    filter(nicotine_dependence == TRUE)
nrow(cohort_nicotine)

In [ ]:
# Opioid dependence cohort for PheWAS
cohort_opioid <- cohort_full %>%
    filter(opioid_dependence == TRUE)
nrow(cohort_opioid)

In [ ]:
# No dependence cohort for PheWAS
cohort_no_sud <- cohort_full %>%
    mutate(n_dependence = rowSums(across(alcohol_dependence:opioid_dependence))) %>%
    filter(n_dependence == 0)
nrow(cohort_no_sud)

In [ ]:
# Save dependence cohorts to bucket
# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- cohort_no_sud

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- '011626_cohort_no_sud.csv'

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)
